# 06 — подготовка данных для LSTM v2

Этот ноутбук заменяет старый `06_LSTM_Data_Preparation.ipynb`.

Что меняется относительно прошлой версии:

- сохраняется `history_length.npy`, чтобы модель **не считала pre-first-seen padding реальной историей**;
- сохраняется `user_index.npy` и общий `all_user_ids.npy` для `user_id` embedding;
- к static добавлены **absolute-time** и **future-horizon calendar** признаки;
- sequence по-прежнему компактный: на диске храним только базовые дневные каналы, rolling/ratio/short-intent считаются в `07` на батче;
- добавлена QA-проверка согласованности `user_id ↔ user_index`, shapes и target.

Временная постановка не меняется: признаки используют только историю до cutoff включительно, target — GMV следующих 30 дней.

In [1]:
from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
import pyarrow.dataset as ds


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "train.parquet").exists():
            return candidate
    raise FileNotFoundError("Не найден data/train.parquet")


PROJECT_ROOT = find_project_root()
INPUT_PATH = PROJECT_ROOT / "data" / "train.parquet"
PREPARED_PATH = PROJECT_ROOT / "data" / "Prepared_data.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data" / "lstm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FORMAT_VERSION = 2
SEQ_LEN = 90
HORIZON = 30

LABELED_CUTOFFS = pd.to_datetime([
    "2025-04-19", "2025-05-19", "2025-06-18", "2025-07-18",
    "2025-08-17", "2025-09-16", "2025-10-16", "2025-11-15",
    "2025-12-15", "2026-01-14",
])
INFERENCE_CUTOFF = pd.Timestamp("2026-02-13")
ALL_CUTOFFS = [*LABELED_CUTOFFS, INFERENCE_CUTOFF]
TIME_ORIGIN = LABELED_CUTOFFS[0]

BASE_SEQUENCE_FEATURES = [
    "search", "cat", "searches",
    "search_to_cart", "search_to_ord", "cat_to_cart", "cat_to_ord",
    "to_cart", "to_ord", "gmv_search", "gmv_cat", "gmv",
    "active",
]
RAW_FEATURES = BASE_SEQUENCE_FEATURES[:-1]
BINARY_FEATURES = {"search", "cat"}
LOG_FEATURES = [name for name in RAW_FEATURES if name not in BINARY_FEATURES]
RAW_COLUMNS = ["event_date", "user_id", *RAW_FEATURES]

CALENDAR_SEQUENCE_FEATURES = [
    "dow_sin", "dow_cos",
    "dom_sin", "dom_cos",
    "doy_sin", "doy_cos",
]

EXTRA_STATIC_FEATURES = [
    "cutoff_doy_sin", "cutoff_doy_cos",
    "cutoff_week_sin", "cutoff_week_cos",
    "cutoff_time_years",
    "forecast_mid_doy_sin", "forecast_mid_doy_cos",
    "forecast_end_doy_sin", "forecast_end_doy_cos",
    "forecast_weekend_fraction",
    "forecast_holiday_days_fraction",
    "forecast_holiday_kernel_mean",
]

# Не target-информация: это заранее известный календарь.
# Берём не отдельный флаг под конкретный финальный праздник, а единый smooth signal,
# который встречается и в train horizons (майские, 01.09, 11.11, Новый год).
COMMERCIAL_HOLIDAY_MONTH_DAY = [
    (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8),
    (2, 14), (2, 23), (3, 8),
    (5, 1), (5, 9), (6, 12),
    (9, 1), (11, 11), (12, 31),
]
HOLIDAY_KERNEL_SIGMA_DAYS = 5.0

print("PROJECT_ROOT:", PROJECT_ROOT)
print("format version:", FORMAT_VERSION)

PROJECT_ROOT: /Users/pinta/Dev/E-CUP-2026
format version: 2


## User universe и first-seen

`user_id` не подаётся в сеть как число. В `07` он используется только как индекс в `nn.Embedding`.

`history_length` нужен для корректного pooling/packing: первые нули в 90-дневном тензоре могут означать, что пользователь ещё не появился в данных, а не реальный неактивный день.

In [2]:
first_seen = pd.read_parquet(
    INPUT_PATH,
    columns=["user_id", "event_date"],
    engine="pyarrow",
)
first_seen["event_date"] = pd.to_datetime(first_seen["event_date"])
first_seen = first_seen.groupby("user_id", sort=True)["event_date"].min()

ALL_USERS = first_seen.index.to_numpy(dtype=np.int64)
np.save(OUTPUT_DIR / "all_user_ids.npy", ALL_USERS)

print("users:", len(ALL_USERS))
print("first seen:", first_seen.min().date(), "->", first_seen.max().date())

users: 250000
first seen: 2025-01-01 -> 2025-12-15


## Базовые 90-дневные последовательности

Формат `X.npy` остаётся совместимым со старой версией: `float16`, 13 каналов. Новые вещи (`history_length`, `user_index`) хранятся отдельно и занимают мало места.

In [3]:
def build_base_cutoff(cutoff):
    cutoff = pd.Timestamp(cutoff)
    cutoff_key = cutoff.strftime("%Y-%m-%d")
    cutoff_dir = OUTPUT_DIR / cutoff_key
    cutoff_dir.mkdir(parents=True, exist_ok=True)

    required = [
        cutoff_dir / "X.npy",
        cutoff_dir / "user_id.npy",
        cutoff_dir / "user_index.npy",
        cutoff_dir / "history_length.npy",
    ]
    if cutoff != INFERENCE_CUTOFF:
        required.append(cutoff_dir / "y.npy")

    # X не менял формат, но для чистого run лучше удалить data/lstm перед запуском.
    # Если все v2-файлы уже есть — не перечитываем parquet.
    if all(path.exists() for path in required):
        print(cutoff.date(), "-- v2 base files already exist, reuse")
        return

    history_start = cutoff - pd.Timedelta(days=SEQ_LEN - 1)
    is_inference = cutoff == INFERENCE_CUTOFF
    read_end = cutoff if is_inference else cutoff + pd.Timedelta(days=HORIZON)

    eligible = first_seen <= cutoff
    users = first_seen.index[eligible].to_numpy(dtype=np.int64)
    first_seen_users = first_seen.loc[users].to_numpy(dtype="datetime64[D]")
    user_index = np.searchsorted(ALL_USERS, users).astype(np.int32)

    first_valid = np.maximum(
        first_seen_users,
        np.datetime64(history_start.date(), "D"),
    )
    history_length = (
        np.datetime64(cutoff.date(), "D") - first_valid
    ).astype("timedelta64[D]").astype(np.int32) + 1
    history_length = np.clip(history_length, 1, SEQ_LEN).astype(np.uint8)

    user_to_row = pd.Series(np.arange(len(users), dtype=np.int32), index=users)

    frame = pd.read_parquet(
        INPUT_PATH,
        columns=RAW_COLUMNS,
        engine="pyarrow",
        filters=[
            ("event_date", ">=", history_start),
            ("event_date", "<=", read_end),
        ],
    )
    frame["event_date"] = pd.to_datetime(frame["event_date"])

    history = frame[frame["event_date"] <= cutoff].copy()
    user_rows = history["user_id"].map(user_to_row)
    valid_rows = user_rows.notna().to_numpy()
    history = history.loc[valid_rows]
    user_rows = user_rows.loc[valid_rows].to_numpy(dtype=np.int32)
    day_index = (history["event_date"] - history_start).dt.days.to_numpy(dtype=np.int32)

    X = np.zeros((len(users), SEQ_LEN, len(BASE_SEQUENCE_FEATURES)), dtype=np.float16)

    for j, name in enumerate(RAW_FEATURES):
        values = history[name].to_numpy(dtype=np.float32)
        if name in LOG_FEATURES:
            values = np.log1p(np.clip(values, 0, None))
        X[user_rows, day_index, j] = values.astype(np.float16)

    X[user_rows, day_index, -1] = 1.0

    np.save(cutoff_dir / "X.npy", X)
    np.save(cutoff_dir / "user_id.npy", users)
    np.save(cutoff_dir / "user_index.npy", user_index)
    np.save(cutoff_dir / "history_length.npy", history_length)

    if not is_inference:
        future = frame[(frame["event_date"] > cutoff) & frame["user_id"].isin(users)]
        y = (
            future.groupby("user_id")["gmv"].sum()
            .reindex(users, fill_value=0.0)
            .to_numpy(dtype=np.float32)
        )
        np.save(cutoff_dir / "y.npy", y)

    print(
        cutoff.date(),
        "users:", len(users),
        "full-history:", f"{(history_length == SEQ_LEN).mean():.1%}",
    )

    del frame, history, X


for cutoff in ALL_CUTOFFS:
    build_base_cutoff(cutoff)

2025-04-19 users: 216457 full-history: 82.0%
2025-05-19 users: 221154 full-history: 90.8%
2025-06-18 users: 225245 full-history: 93.4%
2025-07-18 users: 229146 full-history: 94.5%
2025-08-17 users: 232977 full-history: 95.0%
2025-09-16 users: 236668 full-history: 95.2%
2025-10-16 users: 240700 full-history: 95.2%
2025-11-15 users: 244983 full-history: 95.2%
2025-12-15 users: 250000 full-history: 94.7%
2026-01-14 users: 250000 full-history: 96.3%
2026-02-13 users: 250000 full-history: 98.1%


## Static features

Берём 91 признак из `Prepared_data.parquet` и добавляем только LSTM-specific календарные признаки.

`forecast_*` используют **только календарь следующих 30 дней**, который известен в момент прогноза. Никаких будущих пользовательских событий здесь нет.

In [4]:
prepared_ds = ds.dataset(PREPARED_PATH, format="parquet")
PREPARED_FEATURES = [
    name for name in prepared_ds.schema.names
    if name not in {"user_id", "cutoff_date", "target_gmv_30d", "target_nonzero"}
]

STATIC_FEATURES = [*PREPARED_FEATURES, *EXTRA_STATIC_FEATURES]


def static_log_copy_candidates(feature_names):
    include = ("gmv", "searches", "items", "days", "customer_age", "gap")
    exclude = (
        "trend", "share", "rate", "ratio", "velocity", "cv",
        "sin", "cos", "score", "frequency", "fraction", "kernel",
    )
    return [
        name for name in feature_names
        if any(token in name for token in include)
        and not any(token in name for token in exclude)
    ]


STATIC_LOG_COPY_FEATURES = static_log_copy_candidates(STATIC_FEATURES)


def _annual_holiday_dates(year_min, year_max):
    values = []
    for year in range(year_min, year_max + 1):
        for month, day in COMMERCIAL_HOLIDAY_MONTH_DAY:
            values.append(pd.Timestamp(year=year, month=month, day=day))
    return pd.DatetimeIndex(values)


HOLIDAY_DATES = _annual_holiday_dates(2024, 2027)


def forecast_calendar_features(cutoff):
    cutoff = pd.Timestamp(cutoff)
    future = pd.date_range(cutoff + pd.Timedelta(days=1), periods=HORIZON, freq="D")
    midpoint = future[len(future) // 2]
    end = future[-1]

    # Smooth proximity to the nearest commercial-calendar anchors.
    future_days = future.values.astype("datetime64[D]").astype(np.int64)
    holiday_days = HOLIDAY_DATES.values.astype("datetime64[D]").astype(np.int64)
    distance = np.abs(future_days[:, None] - holiday_days[None, :])
    nearest = distance.min(axis=1)
    kernel = np.exp(-(nearest ** 2) / (2 * HOLIDAY_KERNEL_SIGMA_DAYS ** 2))

    exact_holiday = np.isin(future.normalize(), HOLIDAY_DATES.normalize())

    return np.array([
        np.sin(2 * np.pi * cutoff.dayofyear / 365.25),
        np.cos(2 * np.pi * cutoff.dayofyear / 365.25),
        np.sin(2 * np.pi * int(cutoff.isocalendar().week) / 52.0),
        np.cos(2 * np.pi * int(cutoff.isocalendar().week) / 52.0),
        (cutoff - TIME_ORIGIN).days / 365.25,
        np.sin(2 * np.pi * midpoint.dayofyear / 365.25),
        np.cos(2 * np.pi * midpoint.dayofyear / 365.25),
        np.sin(2 * np.pi * end.dayofyear / 365.25),
        np.cos(2 * np.pi * end.dayofyear / 365.25),
        (future.dayofweek >= 5).mean(),
        exact_holiday.mean(),
        kernel.mean(),
    ], dtype=np.float32)


def sequence_calendar(cutoff):
    dates = pd.date_range(cutoff - pd.Timedelta(days=SEQ_LEN - 1), cutoff, freq="D")
    dow = dates.dayofweek.to_numpy()
    dom = dates.day.to_numpy()
    doy = dates.dayofyear.to_numpy()

    return np.column_stack([
        np.sin(2 * np.pi * dow / 7.0), np.cos(2 * np.pi * dow / 7.0),
        np.sin(2 * np.pi * (dom - 1) / 31.0), np.cos(2 * np.pi * (dom - 1) / 31.0),
        np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25),
    ]).astype(np.float32)


def build_static_cutoff(cutoff):
    cutoff = pd.Timestamp(cutoff)
    cutoff_dir = OUTPUT_DIR / cutoff.strftime("%Y-%m-%d")
    users = np.load(cutoff_dir / "user_id.npy", mmap_mode="r")

    table = prepared_ds.to_table(
        columns=["user_id", *PREPARED_FEATURES],
        filter=ds.field("cutoff_date") == cutoff.date(),
    )
    snapshot = table.to_pandas().set_index("user_id").reindex(np.asarray(users))

    missing_rows = snapshot[PREPARED_FEATURES].isna().all(axis=1)
    if missing_rows.any():
        raise AssertionError(
            f"{cutoff.date()}: {missing_rows.sum()} LSTM users отсутствуют в Prepared_data.parquet"
        )

    static = snapshot[PREPARED_FEATURES].to_numpy(dtype=np.float32)
    extra = np.broadcast_to(
        forecast_calendar_features(cutoff),
        (len(static), len(EXTRA_STATIC_FEATURES)),
    ).copy()
    static = np.concatenate([static, extra], axis=1)

    np.save(cutoff_dir / "static.npy", static.astype(np.float32))
    np.save(cutoff_dir / "calendar.npy", sequence_calendar(cutoff))

    print(cutoff.date(), "static:", static.shape)


for cutoff in ALL_CUTOFFS:
    build_static_cutoff(cutoff)

2025-04-19 static: (216457, 103)
2025-05-19 static: (221154, 103)
2025-06-18 static: (225245, 103)
2025-07-18 static: (229146, 103)
2025-08-17 static: (232977, 103)
2025-09-16 static: (236668, 103)
2025-10-16 static: (240700, 103)
2025-11-15 static: (244983, 103)
2025-12-15 static: (250000, 103)
2026-01-14 static: (250000, 103)
2026-02-13 static: (250000, 103)


## QA и metadata

После этого `07_LSTM.ipynb` может считать data layout фиксированным. Если меняется этот формат — увеличиваем `FORMAT_VERSION`.

In [5]:
def verify_cutoff(cutoff):
    cutoff = pd.Timestamp(cutoff)
    path = OUTPUT_DIR / cutoff.strftime("%Y-%m-%d")

    X = np.load(path / "X.npy", mmap_mode="r")
    static = np.load(path / "static.npy", mmap_mode="r")
    calendar = np.load(path / "calendar.npy", mmap_mode="r")
    users = np.load(path / "user_id.npy", mmap_mode="r")
    user_index = np.load(path / "user_index.npy", mmap_mode="r")
    lengths = np.load(path / "history_length.npy", mmap_mode="r")

    n = len(users)
    assert X.shape == (n, SEQ_LEN, len(BASE_SEQUENCE_FEATURES))
    assert static.shape == (n, len(STATIC_FEATURES))
    assert calendar.shape == (SEQ_LEN, len(CALENDAR_SEQUENCE_FEATURES))
    assert user_index.shape == (n,)
    assert lengths.shape == (n,)
    assert np.all((lengths >= 1) & (lengths <= SEQ_LEN))
    assert np.array_equal(ALL_USERS[np.asarray(user_index)], np.asarray(users))
    assert np.all(np.diff(np.asarray(users)) > 0)
    assert not np.isinf(np.asarray(static)).any()

    if cutoff != INFERENCE_CUTOFF:
        y = np.load(path / "y.npy", mmap_mode="r")
        assert y.shape == (n,)
        assert np.all(np.asarray(y) >= 0)

    return {
        "cutoff": cutoff.strftime("%Y-%m-%d"),
        "users": n,
        "full_history_share": float((np.asarray(lengths) == SEQ_LEN).mean()),
    }


qa = [verify_cutoff(cutoff) for cutoff in ALL_CUTOFFS]
qa_df = pd.DataFrame(qa)
display(qa_df)

meta = {
    "format_version": FORMAT_VERSION,
    "seq_len": SEQ_LEN,
    "horizon": HORIZON,
    "base_sequence_features": BASE_SEQUENCE_FEATURES,
    "calendar_sequence_features": CALENDAR_SEQUENCE_FEATURES,
    "prepared_static_features": PREPARED_FEATURES,
    "extra_static_features": EXTRA_STATIC_FEATURES,
    "static_features": STATIC_FEATURES,
    "static_log_copy_features": STATIC_LOG_COPY_FEATURES,
    "labeled_cutoffs": [x.strftime("%Y-%m-%d") for x in LABELED_CUTOFFS],
    "inference_cutoff": INFERENCE_CUTOFF.strftime("%Y-%m-%d"),
    "user_count": int(len(ALL_USERS)),
    "history_length_file": "history_length.npy",
    "user_index_file": "user_index.npy",
    "base_transform": "log1p for non-binary daily channels; active is 0/1",
    "target": "sum(gmv) for cutoff < event_date <= cutoff + 30 days",
    "future_calendar_note": "calendar-only features; no future user behavior",
}

with open(OUTPUT_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("saved:", OUTPUT_DIR / "meta.json")
print("base sequence:", len(BASE_SEQUENCE_FEATURES))
print("calendar sequence:", len(CALENDAR_SEQUENCE_FEATURES))
print("static on disk:", len(STATIC_FEATURES))
print("static log copies in 07:", len(STATIC_LOG_COPY_FEATURES))

,cutoff,users,full_history_share
0,2025-04-19,216457,0.819844
1,2025-05-19,221154,0.908123
2,2025-06-18,225245,0.934289
3,2025-07-18,229146,0.945341
4,2025-08-17,232977,0.949914
5,2025-09-16,236668,0.952233
6,2025-10-16,240700,0.952381
7,2025-11-15,244983,0.951568
8,2025-12-15,250000,0.947200
9,2026-01-14,250000,0.963288


saved: /Users/pinta/Dev/E-CUP-2026/data/lstm/meta.json
base sequence: 13
calendar sequence: 6
static on disk: 103
static log copies in 07: 51
